## Changing response functions
*R.A. Collenteur, University of Graz, 2021*

In this notebook the new `ChangeModel` is tested, based on the work by [Obergfjell et al. (2019)](https://ngwa.onlinelibrary.wiley.com/doi/10.1111/gwat.12891). The main idea is to apply different response functions for two different periods. As an example we look at the the groundwater levels measured near the river the Mur in Austria, where a dam was recently built. 



In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import pastas as ps

ps.set_log_level("ERROR")
ps.show_versions()

## 1. Load the data

In [ ]:
prec = pd.read_csv("data_step/prec.csv", index_col=0, parse_dates=True).squeeze()
evap = pd.read_csv("data_step/evap.csv", index_col=0, parse_dates=True).squeeze()
head = pd.read_csv("data_step/head.csv", index_col=0, parse_dates=True).squeeze()
river = pd.read_csv("data_step/river.csv", index_col=0, parse_dates=True).squeeze()
river -= river.min()

axes = ps.plots.series(
    head=head,
    stresses=[prec, evap, river],
    tmin="2000",
    labels=["Head\n[m]", "Prec\n[mm/d]", "Evap\n[mm/d]", "River [m3/d]"],
)

## 2. The weighting factor

The stress is convolved two times with different response functions. Then, a weighting function is used to add the two contributions together and compute the final contribution. 

In [ ]:
npoints = 100

tchange = 50 / npoints
t = np.linspace(0, 1, npoints)
color = plt.cm.viridis(np.linspace(0, 1, 10))

for beta, c in zip(np.linspace(-1, 1, 10), color):
    beta1 = beta * npoints
    omega = 1 / (np.exp(beta1 * (t - tchange)) + 1)
    plt.plot(omega, color=c, label=f"$beta$={beta:.2f}")

plt.ylabel(r"$\omega$ [-]")
plt.xlabel("Time [t]")
plt.legend()

## 3. Make a model
We now make two models:

- one model called `"same"` where we assume the response of the heads to the river level remains the same
- one model called `"change"` where the response to the river levels changes. 


In [ ]:
t_step = pd.Timestamp("2012-01-01")
tmin = pd.Timestamp("2004-01-01")
tmax = pd.Timestamp("2017-12-31")

In [ ]:
# Normal Model
ml = ps.Model(head, name="linear")
ps.ArNoiseModel(model=ml)

sm = ps.StressModel(model=ml, stress=river, rfunc=ps.Exponential(), name="test")
step = ps.StepModel(model=ml, tstart="2012-01-01", rfunc=ps.One(), name="step")

ml.solve(report=False, tmin="2004", tmax="2017-12-31")
ml.plots.results(figsize=(10, 6))

# ChangeModel
ml2 = ps.Model(head, name="linear")
ps.ArNoiseModel(model=ml2)

cm = ps.ChangeModel(
    model=ml2,
    stress=river,
    rfunc1=ps.Exponential(),
    rfunc2=ps.Exponential(),
    name="river",
    tchange="2012-01-01",
)
step = ps.StepModel(model=ml2, tstart="2012-01-01", rfunc=ps.One(), name="step")

ml2.solve(report=False, tmin="2004", tmax="2017-12-31")
ml2.plots.results(figsize=(10, 6));

The second model shows a better fit, but also the step trend changed.  

In [ ]:
print("RMSE for the first model:", ml.stats.rmse().round(2))
print("RMSE for the second model:", ml2.stats.rmse().round(2))

## 4. Compare the response functions
We can also look at the response to the river before and after, 

In [ ]:
responses = ml2.get_step_response("river")

f, ax = plt.subplots()
ax.plot(responses.iloc[:, 0], label="Before (ChangeModel)")
ax.plot(responses.iloc[:, 1], label="After (ChangeModel)")
ax.legend()

## 5. Another way
We can also add the stress twice, saving one parameter that needs to be estimated.

In [ ]:
ml3 = ps.Model(head, name="linear")
ps.ArNoiseModel(model=ml3)

river1 = river.copy()
river1.loc["2012":] = 0

river2 = river.copy()
river2.loc[:"2011"] = 0

r1 = ps.StressModel(model=ml3, stress=river1, rfunc=ps.Exponential(), name="river")
r2 = ps.StressModel(model=ml3, stress=river2, rfunc=ps.Exponential(), name="river2")
step = ps.StepModel(model=ml3, tstart="2012-01-01", rfunc=ps.One(), name="step")

ml3.solve(report=False, tmin="2004", tmax="2017-12-31")
_ = ml3.plots.results(figsize=(10, 6))

## How do the results compare?

In [ ]:
f, ax = plt.subplots()

# change model
ax.plot(responses.iloc[:, 0], label="Before (ChangeModel)")
ax.plot(responses.iloc[:, 1], label="After (ChangeModel)")

# 2 stressmodels
ml3.get_step_response("river").plot(ax=ax, label="Before (Two Stresses Method)")
ml3.get_step_response("river2").plot(ax=ax, label="After (Two Stresses Method)")

ax.legend()

## References

Obergfell, C., Bakker, M. and Maas, K. (2019), Identification and Explanation of a Change in the Groundwater Regime using Time Series Analysis. Groundwater, 57: 886-894. https://doi.org/10.1111/gwat.12891